# Multi-Party Systems

In this lab, we will explore next speaker prediction, a key task in real-time and multi-party systems. The goal is to predict who will speak next in a meeting. We will use a Bidirectional Long Short-Term Memory (Bi-LSTM) neural network to tackle this prediction task. For our experiments, we will work with the AMI Meeting Corpus dataset.

Let's begin by installing the necessary dependencies.

In [ ]:
import os
import re
import glob
import copy
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# AMI Dataset
The AMI Meeting Corpus is a publicly released dataset, maintained and distributed by the University of Edinburgh. It is a corpus of meeting recordings with many speakers participating, containing both audio and annotation files. In this experiment, we focus on the transcriptions and speaker information in the AMI annotations, rather than the raw audio signals.

In [ ]:
# download dataset
import os
import subprocess

ami_dir = "ami"
zip_path = os.path.join(ami_dir, "ami_annotations.zip")
url = "https://groups.inf.ed.ac.uk/ami/AMICorpusAnnotations/ami_public_manual_1.6.2.zip"

if os.path.isdir(ami_dir):
    print("ami directory already exists, skipping download and unzip.")
else:
    print("Creating ami directory and downloading/unzipping...")
    os.makedirs(ami_dir, exist_ok=True)
    subprocess.run([
        "wget", "-O", zip_path, url
    ], check=True)
    subprocess.run([
        "unzip", zip_path, "-d", ami_dir
    ], check=True)

Creating ami directory and downloading/unzipping...


In [ ]:
DATAROOT = "ami"
def strip_ns(tag):
    return tag.split("}")[-1] if "}" in tag else tag

def parse_words_xml(xml_path):
    fname = os.path.basename(xml_path)
    m = re.match(r"([A-Z]{2}\d{4}[a-z])\.([A-Z])\.words\.xml", fname)
    if m is None:
        return pd.DataFrame()

    meeting_id, speaker = m.group(1), m.group(2)
    rows = []

    root = ET.parse(xml_path).getroot()
    for elem in root.iter():
        if strip_ns(elem.tag).lower() != "w":
            continue

        attrs = elem.attrib
        start = attrs.get("startTime") or attrs.get("starttime")
        end   = attrs.get("endTime") or attrs.get("endtime")
        text  = (elem.text or "").strip()

        if start is None or end is None or text == "":
            continue

        rows.append({
            "meeting_id": meeting_id,
            "speaker": speaker,
            "start": float(start),
            "end": float(end),
            "text": text,
        })

    return pd.DataFrame(rows)

word_xmls = sorted(glob.glob(os.path.join(DATAROOT, "**", "*.words.xml"), recursive=True))
if len(word_xmls) == 0:
    raise ValueError(f"No .words.xml files found, please check DATAROOT={DATAROOT}")

word_dfs = [parse_words_xml(p) for p in word_xmls]
word_dfs = [df for df in word_dfs if len(df) > 0]

if len(word_dfs) == 0:
    raise ValueError("No valid words data after parsing, please check XML filename format and content.")

words_df = pd.concat(word_dfs, ignore_index=True)
words_df = words_df[words_df["text"] != ""].reset_index(drop=True)

print("words_df:", words_df.shape)

words_df: (986960, 5)


# Turn Building

We construct dialogue turns from the dataset for subsequent model training and inference. Specifically, we first sort the utterances by timestamp to determine the speaker order, and then merge consecutive text segments from the same speaker when the gap between them is smaller than a predefined hyperparameter, MAX_GAP.

In [ ]:
MAX_GAP = 0.5
def build_turns(words_df, max_gap=0.5):
    rows = []

    for mid, sub in words_df.groupby("meeting_id"):
        sub = sub.sort_values(["start", "end"]).reset_index(drop=True)

        cur_speaker = None
        cur_words = []
        cur_start = None
        cur_end = None

        for _, row in sub.iterrows():
            spk = row["speaker"]
            start = row["start"]
            end = row["end"]
            text = row["text"]

            if cur_speaker is None:
                cur_speaker = spk
                cur_start = start
                cur_end = end
                cur_words = [text]
            else:
                if spk == cur_speaker and (start - cur_end) <= max_gap:
                    cur_words.append(text)
                    cur_end = end
                else:
                    rows.append({
                        "meeting_id": mid,
                        "speaker": cur_speaker,
                        "utt_start": cur_start,
                        "utt_end": cur_end,
                        "utt_text": " ".join(cur_words)
                    })
                    cur_speaker = spk
                    cur_start = start
                    cur_end = end
                    cur_words = [text]

        if cur_speaker is not None:
            rows.append({
                "meeting_id": mid,
                "speaker": cur_speaker,
                "utt_start": cur_start,
                "utt_end": cur_end,
                "utt_text": " ".join(cur_words)
            })

    turn_df = pd.DataFrame(rows)
    turn_df = turn_df.sort_values(["meeting_id", "utt_start", "utt_end"]).reset_index(drop=True)
    return turn_df

turn_df = build_turns(words_df, max_gap=MAX_GAP)
print("turn_df:", turn_df.shape)

turn_df: (212312, 5)


In [ ]:
# We only keep 4 speakers
valid_meetings = (
    turn_df.groupby("meeting_id")["speaker"]
    .apply(lambda s: set(s.unique()) == {"A", "B", "C", "D"})
)

valid_meeting_ids = valid_meetings[valid_meetings].index.tolist()
turn_df = turn_df[turn_df["meeting_id"].isin(valid_meeting_ids)].copy()
turn_df = turn_df.reset_index(drop=True)

print("4-speaker turn_df:", turn_df.shape)
print("num meetings:", turn_df["meeting_id"].nunique())

4-speaker turn_df: (197482, 5)
num meetings: 147


In [ ]:
# History turns
HISTORY = 8

SPEAKER2ID = {"A": 0, "B": 1, "C": 2, "D": 3}
SPEAKER_VOCAB = {"A": 0, "B": 1, "C": 2, "D": 3}
def build_dataset(turn_df, history=8):
    rows = []

    for mid, sub in turn_df.groupby("meeting_id"):
        sub = sub.sort_values("utt_start").reset_index(drop=True)

        for i in range(history, len(sub) - 1):
            cur_row = sub.iloc[i]
            next_row = sub.iloc[i + 1]

            history_texts = [sub.iloc[j].utt_text for j in range(i - history, i)]
            history_speakers = [sub.iloc[j].speaker for j in range(i - history, i)]

            rows.append({
                "meeting_id": mid,
                "speaker": cur_row.speaker,
                "text": cur_row.utt_text,
                "history_texts": history_texts,
                "history_speakers": history_speakers,
                "next_speaker": next_row.speaker,
                "label": SPEAKER2ID[next_row.speaker],
            })

    return pd.DataFrame(rows)

dataset = build_dataset(turn_df, history=HISTORY)

if len(dataset) == 0:
    raise ValueError("Constructed dataset is empty, possibly HISTORY too large or data parsing issue.")

print("dataset:", dataset.shape)
print(dataset.head(2))

dataset: (196159, 7)
  meeting_id speaker             text  \
0    EN2001b       A           Nice .   
1    EN2001b       B  Group K_ . Da .   
                                       history_texts  \
0  [Hmm, Yeah ., It's with cameras, Hmm ?, ., You...   
1  [Yeah ., It's with cameras, Hmm ?, ., You look...   
           history_speakers next_speaker  label  
0  [A, C, A, B, A, A, B, B]            B      1  
1  [C, A, B, A, A, B, B, A]            B      1  


# Dataset Splitting
Since this dataset has not been pre-split, we need to divide it into training, development, and test sets in specific proportions for subsequent training and evaluation.

In [ ]:
meetings = sorted(dataset["meeting_id"].unique())

# Set a random seed so that the train/dev/test split is reproducible
np.random.seed(42)
np.random.shuffle(meetings)

n = len(meetings)
train_ids = meetings[:int(0.8 * n)]
dev_ids = meetings[int(0.8 * n):int(0.9 * n)]
test_ids = meetings[int(0.9 * n):]

train_df = dataset[dataset["meeting_id"].isin(train_ids)].reset_index(drop=True)
dev_df = dataset[dataset["meeting_id"].isin(dev_ids)].reset_index(drop=True)
test_df = dataset[dataset["meeting_id"].isin(test_ids)].reset_index(drop=True)

for df in [train_df, dev_df, test_df]:
    df["speaker_id"] = df["speaker"].map(SPEAKER_VOCAB)
    df["history_speaker_ids"] = df["history_speakers"].apply(
        lambda spks: [SPEAKER_VOCAB[s] for s in spks]
    )

print("train/dev/test:", len(train_df), len(dev_df), len(test_df))

train/dev/test: 161108 13389 21662


In [ ]:
def simple_tokenize(text):
    text = re.sub(r"[^\w\s']", " ", str(text).lower().strip())
    text = re.sub(r"\s+", " ", text)
    return text.split()

counter = Counter()

for _, row in train_df.iterrows():
    counter.update(simple_tokenize(row["text"]))
    for h in row["history_texts"]:
        counter.update(simple_tokenize(h))

vocab = {"<PAD>": 0, "<UNK>": 1}
for word, freq in counter.items():
    if freq >= 2:
        vocab[word] = len(vocab)

def encode_text(text):
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in simple_tokenize(text)]
    return ids if len(ids) > 0 else [vocab["<UNK>"]]

print("vocab size:", len(vocab))

vocab size: 10531


# Dataset Examples
We print out some dataset examples to get a better sense of what the data looks like.

In [ ]:
def print_words_examples(words_df, n=10, meeting_id=None):
    print("\n" + "="*120)
    print("WORDS EXAMPLES")
    print("="*120)

    if meeting_id is None:
        print(words_df.head(n).to_string(index=False))
    else:
        sub = words_df[words_df["meeting_id"] == meeting_id].sort_values("start").reset_index(drop=True)
        print(sub.head(n).to_string(index=False))

def print_turn_examples(turn_df, n=20, meeting_id=None):
    print("\n" + "="*120)
    print("TURN EXAMPLES")
    print("="*120)

    if meeting_id is None:
        sub = turn_df.head(n).copy()
    else:
        sub = (
            turn_df[turn_df["meeting_id"] == meeting_id]
            .sort_values("utt_start")
            .reset_index(drop=True)
            .head(n)
            .copy()
        )

    for i, row in sub.iterrows():
        dur = row["utt_end"] - row["utt_start"]
        print(f"[{i:03d}] {row['meeting_id']} | {row['speaker']} | "
              f"{row['utt_start']:.2f}-{row['utt_end']:.2f} ({dur:.2f}s) | {row['utt_text']}")

def show_dataset_example(df, idx):
    row = df.iloc[idx]

    print("\n" + "="*120)
    print(f"DATASET EXAMPLE idx={idx}")
    print("="*120)
    print(f"meeting_id   : {row['meeting_id']}")
    print(f"cur speaker  : {row['speaker']}")
    print(f"next speaker : {row['next_speaker']}")
    print(f"label        : {row['label']}")
    print("-"*120)
    print("history_texts + history_speakers:")
    for j, (h_spk, h_txt) in enumerate(zip(row["history_speakers"], row["history_texts"]), 1):
        print(f"  H{j}: [{h_spk}] {h_txt}")
    print("-"*120)
    print(f"current text : {row['text']}")
    print("="*120)

def show_random_dataset_examples(df, n=5, seed=42):
    idxs = df.sample(n, random_state=seed).index.tolist()
    for idx in idxs:
        show_dataset_example(df, idx)

def print_label_distribution(df, name="split"):
    print("\n" + "="*120)
    print(f"LABEL DISTRIBUTION: {name}")
    print("="*120)
    counts = df["next_speaker"].value_counts().sort_index()
    ratios = (df["next_speaker"].value_counts(normalize=True).sort_index() * 100).round(2)

    print("counts:")
    print(counts)
    print("\nratios(%):")
    print(ratios)

    same = (df["speaker"] == df["next_speaker"]).mean()
    print(f"\ncurrent speaker == next speaker ratio: {same:.4f}")

In [ ]:
print("\n" + "#"*120)
print("Start printing data examples, check data before training")
print("#"*120)

example_meeting = dataset.iloc[0]["meeting_id"]
print_words_examples(words_df, n=20, meeting_id=example_meeting)
print_turn_examples(turn_df, n=30, meeting_id=example_meeting)

print("\nView 3 random dataset samples:")
show_random_dataset_examples(dataset, n=3, seed=123)

print_label_distribution(train_df, "train")
print_label_distribution(dev_df, "dev")
print_label_distribution(test_df, "test")

########################################################################################################################
Start printing data examples, check data before training
########################################################################################################################
WORDS EXAMPLES
meeting_id speaker  start   end    text
   EN2001b       A   3.09  3.24     Hmm
   EN2001b       C  22.30 22.69    Yeah
   EN2001b       C  22.69 22.69       .
   EN2001b       A  30.60 30.97    It's
   EN2001b       A  30.97 31.11    with
   EN2001b       A  31.11 34.62 cameras
   EN2001b       B  32.19 32.86     Hmm
   EN2001b       B  32.86 32.86       ?
   EN2001b       A  34.62 34.62       .
   EN2001b       A  35.67 35.79     You
   EN2001b       A  35.79 36.18    look
   EN2001b       A  36.18 36.41   quite
   EN2001b       A  36.41 36.69   funny
   EN2001b       A  36.69 36.83      at
   EN2001b       A  36.83 36.89     the
   EN2001b       A  36.89 37.20  moment
   EN2

# Data Loader

We use separate data loaders to load the training, development, and test sets.

In [ ]:
BATCH_SIZE = 16

class NextSpeakerDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "history_tokens": [
                torch.tensor(encode_text(t), dtype=torch.long)
                for t in row["history_texts"]
            ],
            "history_speakers": torch.tensor(row["history_speaker_ids"], dtype=torch.long),
            "tokens": torch.tensor(encode_text(row["text"]), dtype=torch.long),
            "speaker": torch.tensor(int(row["speaker_id"]), dtype=torch.long),
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
        }

def pad_seq(seqs):
    lengths = torch.tensor([len(x) for x in seqs], dtype=torch.long)
    max_len = lengths.max().item()
    padded = torch.zeros(len(seqs), max_len, dtype=torch.long)

    for i, seq in enumerate(seqs):
        padded[i, :len(seq)] = seq

    return padded, lengths

def collate_fn(batch):
    tokens, lengths = pad_seq([b["tokens"] for b in batch])

    history_tokens = []
    history_lengths = []

    for h in range(HISTORY):
        ht, hl = pad_seq([b["history_tokens"][h] for b in batch])
        history_tokens.append(ht)
        history_lengths.append(hl)

    history_speakers = torch.stack([b["history_speakers"] for b in batch])  # [B, H]

    return {
        "tokens": tokens,
        "lengths": lengths,
        "history_tokens": history_tokens,
        "history_lengths": history_lengths,
        "history_speakers": history_speakers,
        "speaker": torch.stack([b["speaker"] for b in batch]),
        "label": torch.stack([b["label"] for b in batch]),
    }

train_loader = DataLoader(
    NextSpeakerDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

dev_loader = DataLoader(
    NextSpeakerDataset(dev_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    NextSpeakerDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
batch = next(iter(train_loader))

print("tokens:", batch["tokens"].shape)
print("lengths:", batch["lengths"].shape)
print("history_speakers:", batch["history_speakers"].shape)
print("speaker:", batch["speaker"].shape)
print("label:", batch["label"].shape)
print("number of history token tensors:", len(batch["history_tokens"]))
if len(batch["history_tokens"]) > 0:
    print("first history token tensor:", batch["history_tokens"][0].shape)
else:
    print("first history token tensor: no history tokens because HISTORY = 0")

tokens: torch.Size([16, 10])
lengths: torch.Size([16])
history_speakers: torch.Size([16, 8])
speaker: torch.Size([16])
label: torch.Size([16])
number of history token tensors: 8
first history token tensor: torch.Size([16, 51])


# Neural Network

We use a bidirectional LSTM to build a model for next-speaker prediction. In this task, the model is designed to take into account the utterance content, speaker information, and dialogue history. In this part, we introduce two variables that control whether speaker information and dialogue history are incorporated into the model. You are asked to implement the corresponding neural network architecture.

In [ ]:
USE_CURRENT_SPEAKER = False
USE_HISTORY_SPEAKER = True

EMB_DIM = 128
WORD_HIDDEN = 128
CTX_HIDDEN = 128
SPK_EMB_DIM = 16
DROPOUT = 0.3

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class UtteranceBiLSTMEncoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, dropout=0.3):
        super().__init__()

        # create an embedding layer
        # embedding size: [vocab_size, emb_dim]
        # Embedding layer converts token IDs into dense word vectors
        # padding_idx=0 means the <PAD> token will not contribute useful information
        self.emb = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=emb_dim,
            padding_idx=0
        )

        # create a 1-layer BİLSTM
        # input size: emb_dim
        # hidden size: hidden_dim
        # A 1-layer bidirectional LSTM reads the utterance in both directions
        # batch_first=True means the input shape is [batch_size, sequence_length, emb_dim]
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Dropout is used to reduce overfitting
        self.dropout = nn.Dropout(dropout)

    def forward(self, tokens, lengths):
        # Convert token IDs into embeddings.
        # Input shape:  [B, T]
        # Output shape: [B, T, emb_dim]
        x = self.emb(tokens)

        # Pack the padded sequences so that the LSTM ignores padding tokens
        # moved lengths to CPU because pack_padded_sequence expects CPU lengths
        packed = nn.utils.rnn.pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # Pass the packed sequence into the BiLSTM
        # h_n contains the final hidden states from the forward and backward LSTMs
        _, (h_n, _) = self.lstm(packed)

        # For a 1-layer BiLSTM:
        # h_n[0] is the final forward hidden state
        # h_n[1] is the final backward hidden state
        h_fw = h_n[0]
        h_bw = h_n[1]

        # Concatenate forward and backward states
        # Final shape: [B, 2 * hidden_dim]
        h = torch.cat([h_fw, h_bw], dim=1)

        # Apply dropout
        h = self.dropout(h)

        return h


class NextSpeakerBiLSTMModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        history=8,
        emb_dim=128,
        word_hidden=128,
        ctx_hidden=128,
        spk_emb_dim=16,
        dropout=0.3,
        use_current_speaker=True,
        use_history_speaker=True,
    ):
        super().__init__()

        self.history = history
        self.use_current_speaker = use_current_speaker
        self.use_history_speaker = use_history_speaker

        # The utterance encoder converts each utterance into a fixed-size vector
        # Since it is bidirectional, the output size is [B, 2 * word_hidden]
        self.utt_encoder = UtteranceBiLSTMEncoder(
            vocab_size=vocab_size,
            emb_dim=emb_dim,
            hidden_dim=word_hidden,
            dropout=dropout
        )

        # A BiLSTM utterance encoder outputs forward + backward hidden states
        base_utt_dim = 2 * word_hidden

        if self.use_history_speaker:
            # Speaker embeddings add information about who produced each previous turn
            # There are 4 possible speakers: A, B, C and D
            self.history_speaker_emb = nn.Embedding(
                num_embeddings=4,
                embedding_dim=spk_emb_dim
            )

            # Each history vector contains the utterance representation plus speaker embedding
            hist_utt_dim = base_utt_dim + spk_emb_dim
        else:
            self.history_speaker_emb = None
            hist_utt_dim = base_utt_dim

        # This is the input size of the context-level LSTM
        self.ctx_input_dim = hist_utt_dim

        # The context BiLSTM reads the sequence of history utterances plus the current utterance
        # It models the dialogue context before predicting the next speaker
        self.ctx_lstm = nn.LSTM(
            input_size=self.ctx_input_dim,
            hidden_size=ctx_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Current speaker embedding is optionally added before classification
        self.spk_emb = nn.Embedding(
            num_embeddings=4,
            embedding_dim=spk_emb_dim
        )

        # Dropout for regularisation
        self.dropout = nn.Dropout(dropout)

        # Context BiLSTM output size is [B, 2 * ctx_hidden]
        clf_in_dim = 2 * ctx_hidden

        if self.use_current_speaker:
            # If current speaker information is used, concatenate its embedding
            clf_in_dim += spk_emb_dim

        # Final classifier maps the dialogue representation to 4 speaker classes
        self.clf = nn.Sequential(
            nn.Linear(clf_in_dim, ctx_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_hidden, 4)
        )

    def forward(self, history_tokens, history_lengths, history_speakers,
                tokens, lengths, speaker):
        hist_vecs = []
        batch_size = tokens.size(0)

        # Encode each history utterance separately
        for h_idx, (ht, hl) in enumerate(zip(history_tokens, history_lengths)):
            # Move tensors to the same device as the model
            ht = ht.to(device)
            hl = hl.to(device)

            # Encode history utterance
            # Shape: [B, 2 * word_hidden]
            hv = self.utt_encoder(ht, hl)

            if self.use_history_speaker:
                # Get the speaker ID for this history position
                # Shape: [B]
                hs = history_speakers[:, h_idx].to(device)

                # Convert speaker IDs into embeddings
                # Shape: [B, spk_emb_dim]
                hs_emb = self.history_speaker_emb(hs)

                # Concatenate utterance vector with speaker embedding
                # Shape: [B, 2 * word_hidden + spk_emb_dim]
                hv = torch.cat([hv, hs_emb], dim=1)

            hist_vecs.append(hv)

        # Encode the current utterance
        # Shape: [B, 2 * word_hidden]
        tokens = tokens.to(device)
        lengths = lengths.to(device)
        cur_vec = self.utt_encoder(tokens, lengths)

        if self.use_history_speaker:
            # The current utterance does not have a history-speaker embedding,
            # but it must have the same dimensionality as the history vectors
            zero_spk = torch.zeros(batch_size, SPK_EMB_DIM, device=device)
            cur_vec = torch.cat([cur_vec, zero_spk], dim=1)

        # Stack history vectors and the current utterance vector into a context sequence.
        # Shape: [B, H+1, ctx_input_dim]
        seq_vecs = torch.stack(hist_vecs + [cur_vec], dim=1)

        # Pass the context sequence through the context-level BiLSTM
        _, (h_n, _) = self.ctx_lstm(seq_vecs)

        # For a 1-layer BiLSTM:
        # h_n[0] is the forward hidden state
        # h_n[1] is the backward hidden state
        ctx_fw = h_n[0]
        ctx_bw = h_n[1]

        # Concatenate both directions to get the final context vector
        # Shape: [B, 2 * ctx_hidden]
        ctx_vec = torch.cat([ctx_fw, ctx_bw], dim=1)

        # Apply dropout
        ctx_vec = self.dropout(ctx_vec)

        if self.use_current_speaker:
            # Add current speaker information before classification
            speaker = speaker.to(device)
            spk_vec = self.spk_emb(speaker)
            x = torch.cat([ctx_vec, spk_vec], dim=1)
        else:
            x = ctx_vec

        # Produce final logits for 4 possible next speakers
        logits = self.clf(x)

        return logits


model = NextSpeakerBiLSTMModel(
    vocab_size=len(vocab),
    history=HISTORY,
    emb_dim=EMB_DIM,
    word_hidden=WORD_HIDDEN,
    ctx_hidden=CTX_HIDDEN,
    spk_emb_dim=SPK_EMB_DIM,
    dropout=DROPOUT,
    use_current_speaker=USE_CURRENT_SPEAKER,
    use_history_speaker=USE_HISTORY_SPEAKER,
).to(device)

print(model)

NextSpeakerBiLSTMModel(
  (utt_encoder): UtteranceBiLSTMEncoder(
    (emb): Embedding(10531, 128, padding_idx=0)
    (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (history_speaker_emb): Embedding(4, 16)
  (ctx_lstm): LSTM(272, 128, batch_first=True, bidirectional=True)
  (spk_emb): Embedding(4, 16)
  (dropout): Dropout(p=0.3, inplace=False)
  (clf): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=4, bias=True)
  )
)


In [ ]:
print("Device:", device)

Device: cuda


# Training

After constructing the neural network, we use an optimiser to train the model from scratch. The hyperparameters can be tuned to achieve optimal performance. During training, we employ early stopping to select the best-performing dev model for later evaluation.

In [ ]:
LR = 3e-5
MAX_EPOCHS = 30
PATIENCE = 5
SEED = 42
MAX_GAP = 0.5

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",      # Monitor dev_f1, higher is better
    factor=0.5,      # Reduce LR by 0.5
    patience=1,      # Reduce after 1 epoch without improvement
    threshold=1e-4,
    min_lr=1e-6,
)

In [ ]:
def run_batch(batch):
    history_tokens = [x.to(device) for x in batch["history_tokens"]]
    history_lengths = [x.to(device) for x in batch["history_lengths"]]
    history_speakers = batch["history_speakers"].to(device)
    tokens = batch["tokens"].to(device)
    lengths = batch["lengths"].to(device)
    speaker = batch["speaker"].to(device)

    logits = model(
        history_tokens=history_tokens,
        history_lengths=history_lengths,
        history_speakers=history_speakers,
        tokens=tokens,
        lengths=lengths,
        speaker=speaker
    )
    return logits

def evaluate(loader):
    model.eval()
    all_y, all_p = [], []

    with torch.no_grad():
        for batch in loader:
            logits = run_batch(batch)
            preds = logits.argmax(dim=1).cpu().numpy()
            golds = batch["label"].cpu().numpy()

            all_p.extend(preds)
            all_y.extend(golds)

    acc = accuracy_score(all_y, all_p)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_y, all_p, average="macro", zero_division=0
    )
    return acc, precision, recall, f1

In [ ]:
best_f1 = -1.0
best_state = None
no_improve = 0

print("Training begins...")

for epoch in range(MAX_EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()

        logits = run_batch(batch)
        loss = criterion(logits, batch["label"].to(device))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    dev_acc, dev_prec, dev_rec, dev_f1 = evaluate(dev_loader)
    scheduler.step(dev_f1)

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1:02d} | "
        f"loss={avg_loss:.4f} | "
        f"dev_acc={dev_acc:.4f} | "
        f"dev_prec={dev_prec:.4f} | "
        f"dev_rec={dev_rec:.4f} | "
        f"dev_f1={dev_f1:.4f}"
    )

    if dev_f1 > best_f1:
        best_f1 = dev_f1
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

Training begins...
Epoch 01 | loss=1.0495 | dev_acc=0.6652 | dev_prec=0.6600 | dev_rec=0.6596 | dev_f1=0.6598
Epoch 02 | loss=0.9232 | dev_acc=0.6638 | dev_prec=0.6602 | dev_rec=0.6567 | dev_f1=0.6582
Epoch 03 | loss=0.8975 | dev_acc=0.6679 | dev_prec=0.6667 | dev_rec=0.6595 | dev_f1=0.6624
Epoch 04 | loss=0.8878 | dev_acc=0.6677 | dev_prec=0.6645 | dev_rec=0.6598 | dev_f1=0.6618
Epoch 05 | loss=0.8791 | dev_acc=0.6694 | dev_prec=0.6661 | dev_rec=0.6616 | dev_f1=0.6635
Epoch 06 | loss=0.8719 | dev_acc=0.6697 | dev_prec=0.6662 | dev_rec=0.6619 | dev_f1=0.6638
Epoch 07 | loss=0.8651 | dev_acc=0.6710 | dev_prec=0.6664 | dev_rec=0.6647 | dev_f1=0.6655
Epoch 08 | loss=0.8594 | dev_acc=0.6701 | dev_prec=0.6656 | dev_rec=0.6635 | dev_f1=0.6644
Epoch 09 | loss=0.8532 | dev_acc=0.6696 | dev_prec=0.6659 | dev_rec=0.6619 | dev_f1=0.6634
Epoch 10 | loss=0.8473 | dev_acc=0.6714 | dev_prec=0.6689 | dev_rec=0.6630 | dev_f1=0.6655
Epoch 11 | loss=0.8436 | dev_acc=0.6713 | dev_prec=0.6677 | dev_rec=0.6

# Evaluation

We use accuracy, precision, recall, and F1-score to comprehensively evaluate the model’s performance. We also print the predictions to facilitate a more detailed analysis of the model.

In [ ]:
if best_state is None:
    raise ValueError("best_state is None, model not saved successfully during training.")

model.load_state_dict(best_state)
model.eval()

all_y, all_p = [], []

with torch.no_grad():
    for batch in test_loader:
        logits = run_batch(batch)
        preds = logits.argmax(dim=1).cpu().numpy()
        golds = batch["label"].cpu().numpy()

        all_p.extend(preds)
        all_y.extend(golds)

acc = accuracy_score(all_y, all_p)
precision, recall, f1, _ = precision_recall_fscore_support(
    all_y, all_p, average="macro", zero_division=0
)
cm = confusion_matrix(all_y, all_p)

print(f"\nTest Acc={acc:.4f} | Macro-F1={f1:.4f}")
print(f"Macro Precision={precision:.4f} | Macro Recall={recall:.4f}")
print("Confusion Matrix:")
print(cm)

Test Acc=0.6863 | Macro-F1=0.6851
Macro Precision=0.6859 | Macro Recall=0.6846
Confusion Matrix:
[[4329  540  567  532]
 [ 648 3229  559  487]
 [ 636  545 3621  565]
 [ 674  500  542 3688]]


In [ ]:
ID2SPEAKER = {v: k for k, v in SPEAKER2ID.items()}

def print_prediction_examples(model, df, n=10, seed=42, correct_only=False, wrong_only=False):
    model.eval()
    sample = df.sample(n * 5, random_state=seed).reset_index(drop=True)

    shown = 0
    with torch.no_grad():
        for _, row in sample.iterrows():
            if shown >= n:
                break

            # Encode inputs
            history_tokens = [torch.tensor(encode_text(t), dtype=torch.long).unsqueeze(0).to(device)
                              for t in row['history_texts']]
            history_lengths = [torch.tensor([len(encode_text(t))], dtype=torch.long).to(device)
                               for t in row['history_texts']]
            history_speakers = torch.tensor([row['history_speaker_ids']], dtype=torch.long).to(device)
            tokens = torch.tensor(encode_text(row['text']), dtype=torch.long).unsqueeze(0).to(device)
            lengths = torch.tensor([len(encode_text(row['text']))], dtype=torch.long).to(device)
            speaker = torch.tensor([int(row['speaker_id'])], dtype=torch.long).to(device)

            logits = model(
                history_tokens=history_tokens,
                history_lengths=history_lengths,
                history_speakers=history_speakers,
                tokens=tokens,
                lengths=lengths,
                speaker=speaker
            )
            probs = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
            pred_id = int(logits.argmax(dim=1).cpu())
            gold_id = int(row['label'])

            if correct_only and pred_id != gold_id:
                continue
            if wrong_only and pred_id == gold_id:
                continue

            result = "✓ CORRECT" if pred_id == gold_id else "✗ WRONG"
            print("=" * 80)
            print(f"[{result}]  Meeting: {row['meeting_id']}  |  Current Speaker: {row['speaker']}")
            print("-" * 80)
            print("Dialogue History:")
            for j, (spk, txt) in enumerate(zip(row['history_speakers'], row['history_texts']), 1):
                print(f"  H{j} [{spk}]: {txt}")
            print(f"  CUR [{row['speaker']}]: {row['text']}")
            print("-" * 80)
            print(f"  Gold next speaker : {row['next_speaker']} {row['text']}")
            print(f"  Pred next speaker : {ID2SPEAKER[pred_id]}")
            print("  Confidence        : " + "  ".join(
                f"{ID2SPEAKER[i]}={probs[i]:.2f}" for i in range(4)
            ))
            shown += 1
    print("=" * 80)

# Print 10 random examples from the test set
print_prediction_examples(model, test_df, n=10, seed=42)

# Print examples where the model was wrong
print("\n\n>>> Wrong predictions only <<<\n")
print_prediction_examples(model, test_df, n=5, seed=42, wrong_only=True)

# Print examples where the model was correct
print("\n\n>>> Correct predictions only <<<\n")
print_prediction_examples(model, test_df, n=5, seed=42, correct_only=True)

[✗ WRONG]  Meeting: TS3008d  |  Current Speaker: A
--------------------------------------------------------------------------------
Dialogue History:
  H1 [A]: select the twenty uh
  H2 [C]: , but
  H3 [A]: a number in the
  H4 [C]: but
  H5 [A]: twenty
  H6 [C]: a lot
  H7 [A]: or
  H8 [D]: Yes
  CUR [A]: in the
--------------------------------------------------------------------------------
  Gold next speaker : C in the
  Pred next speaker : D
  Confidence        : A=0.19  B=0.07  C=0.26  D=0.47
[✓ CORRECT]  Meeting: IS1001b  |  Current Speaker: D
--------------------------------------------------------------------------------
Dialogue History:
  H1 [A]: . Okay
  H2 [D]: 'Kay you can
  H3 [A]: .
  H4 [D]: go so .
  H5 [D]: So there are other frustrations expressed by users , so they said uh they lost uh often the remote control in in the
  H6 [A]: Yeah
  H7 [D]: room
  H8 [A]: .
  CUR [D]: so they want to have a way to to
-------------------------------------------------------------

# Questions

1. What impact does removing dialogue history or speaker information from the training dataset have on model performance? [4 marks]


    Removing dialogue history and speaker information clearly reduced model performance. The full model performed best with 0.7119 test accuracy and 0.7106 macro-F1. When I removed all speaker information, macro-F1 dropped to 0.2589. This shows that speaker identity is a key part of the task, it's not just an extra feature. Removing dialogue history also caused a large drop. The no-history model reached only 0.2163 macro-F1, and its confusion matrix showed that it almost never predicted speaker C. This shows that the current utterance alone does not provide enough information for balanced next-speaker prediction. The additional run where I kept history speaker information and removed current speaker information reached 0.6851 macro-F1, which is close to the full model. This suggests that the sequence of previous speakers is especially useful.




2. What do you notice about the model performance and how it changes in training? Can you suggest why this happens? [4 marks]


    The full model improved during the early stages of training and then levelled off. Its best development macro-F1 was 0.6966, with early stopping at epoch 16. This suggests that the model learned the main turn-taking patterns relatively early, while later epochs gave only small or inconsistent gains. The ablation models showed much weaker learning. The model without speaker information reached only about 0.2590 dev macro-F1, even though it trained for more epochs. The no-history model stopped much earlier and reached only 0.2421 dev macro-F1. A likely reason is that meeting dialogue often contains short, incomplete or generic utterances like “yeah”, “uh”, “okay”, or sentence fragments. These utterances are difficult to interpret without knowing the surrounding dialogue and the order of speakers.



3. What kinds of dialogue context seem to make the model predict incorrectly? What kind of information (perhaps beyond the text) might help in these situations, and why? [4 marks]


    The model often made incorrect predictions when the dialogue context was short, fragmented, or ambiguous. Many wrong examples involved very short current utterances such as “uh”, “that’s”, “in the”, or short backchannel responses. In these cases, the text itself gives very little information about the next speaker. Errors also appeared in fast exchanges where several speakers contributed short turns close together. In these situations, the next speaker may depend more on interaction patterns than on lexical content. For example, timing, pauses, overlap, interruption, gaze direction or body movement could all affect who speaks next. Audio features and visual cues would therefore likely help because real meeting turn-taking is not based only on transcribed words.



4. How could you leverage large language models (LLMs) for this task? Would it be likely to improve the performance, and why? [4 marks]


    LLMs could be used to provide stronger text representations for this task. For example, each utterance could be encoded with a pretrained language model and then passed into the next-speaker prediction model. Another possible approach would be to fine-tune an LLM directly on the next-speaker prediction objective. This would likely improve the model’s understanding of longer and more meaningful turns, especially where dialogue intent or question-answer structure matters. However, an LLM using only text would still have limitations. Many errors are caused by missing turn-taking information, such as pauses, overlap, timing and gaze. For this reason, LLMs would probably improve the language modelling part of the system, but the strongest approach would combine LLM-based text features with speaker history, timing, audio and possibly visual information.